# Workshop Group Chat: Comité Clínico (Hospital)

## 🎯 ¿Qué es el Patrón Group Chat?

El patrón **Group Chat** modela una **conversación multirol** moderada por un **manager**.
En lugar de “pasar el trabajo” (handoff) o ejecutar en paralelo, aquí los especialistas **deliberan en varias rondas** hasta construir una decisión coordinada.


## 🏥 Escenario del ejercicio (muy concreto)

Se convoca un comité clínico por un evento de alta presión asistencial:
> “ALERTA: llegada masiva de pacientes tras accidente múltiple (12 en 20 min; 2 críticos). UCI con 1 cama, radiología saturada (60 min), hay que activar triaje y reasignar personal.”

Lo que queremos demostrar con **Group Chat** es esto:
1. `Jefe_Guardia` (manager) decide a qué rol preguntar en cada ronda y por qué.
2. `Medico_Urgencias`, `Supervision_Enfermeria` y `Farmacia_Hospitalaria` aportan desde su especialidad.
3. La conversación iterativa converge en un plan coordinado para la próxima hora.

### ✅ Qué deberías ver en la salida
- Mensajes alternando roles, con el nombre del agente en cada intervención.
- Intervenciones del manager en formato de ruteo (`[SIGUIENTE: ...]`, `[RAZÓN: ...]`, `[ANÁLISIS: ...]`).
- Un plan que integra clínica + operación + medicación, orientado a acciones inmediatas.

---

## 1) Configuración del Entorno

In [1]:
import os
from dotenv import load_dotenv
from agent_framework import ChatAgent, GroupChatBuilder
from agent_framework.openai import OpenAIChatClient

load_dotenv()

base_url = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
model_id = os.getenv("AZURE_OPENAI_DEPLOYMENT")

print("="*80)
print("✅ Comité Clínico - Group Chat Pattern")
print("="*80)
print(f"\n🏥 Sistema de deliberación clínica:")
print(f"   Endpoint: {'✓' if base_url else '✗'}")
print(f"   API Key:  {'✓' if api_key else '✗'}")
print(f"   Modelo:   {model_id if model_id else '✗'}")

✅ Comité Clínico - Group Chat Pattern

🏥 Sistema de deliberación clínica:
   Endpoint: ✓
   API Key:  ✓
   Modelo:   gpt-5.4


## 2) Concepto: Group Chat vs otros patrones

```
SEQUENTIAL          CONCURRENT          GROUP CHAT         HANDOFF
A → B → C          A┐                   ┌─ A ─┐          A ↔ B
                   B├─ (paralelo)       │     │            ↕
                   C┘                   Manager           C ↔ D
                                        │     │            ↕
Cadena de          Análisis             └─ B ─┘            E
pasos              paralelo             Conversación       Escalado
                                        iterativa          dinámico
```

| Patrón | Flujo | Comunicación | Iteraciones | Mejor para |
|---|---|---|---|---|
| **Sequential** | A→B→C | Lineal | 1 paso c/u | Pipeline fijo |
| **Concurrent** | A,B,C paralelo | Independiente | Sin diálogo | Perspectivas rápidas |
| **Group Chat** | Conversación iterativa | Todos colaboran | N rondas | Consenso/decisión |
| **Handoff** | Escalado dinámico | Punto a punto | Según contexto | Derivar/validar |
| **Magentic** | Plan + ejecución | Plan-driven | Iterativa con plan | Problemas complejos |


## 3) Implementación: Comité Clínico básico

Un manager (Jefe de Guardia) modera la conversación y decide a quién preguntar en cada ronda.

In [2]:
async def comite_clinico():
    """
    Escenario: comité clínico deliberando un incidente asistencial.

    Participantes:
    - Jefe de Guardia (manager): modera y toma la decisión final
    - Médico de Urgencias: perspectiva clínica y priorización
    - Supervisión de Enfermería: capacidad operativa y circuitos
    - Farmacia Hospitalaria: seguridad de medicación y abastecimiento
    """
    
    jefe_guardia = ChatAgent(
        chat_client=OpenAIChatClient(
            base_url=base_url,
            api_key=api_key,
            model_id=model_id,
        ),
        name="Jefe_Guardia",
        instructions="""Eres el Jefe de Guardia moderando un comité clínico.

Tu rol en CADA turno:
1. Analiza lo que se ha dicho hasta ahora
2. Decide si necesitas escuchar a OTRO rol o si ya hay consenso
3. RESPONDE SIEMPRE EN ESTE FORMATO EXACTO:

[SIGUIENTE: nombre_del_rol]
[RAZÓN: explicación breve de por qué lo necesitas]
[ANÁLISIS: síntesis de lo deliberado hasta ahora]

Nombres disponibles:
- Medico_Urgencias
- Supervision_Enfermeria
- Farmacia_Hospitalaria

Ejemplo:
[SIGUIENTE: Farmacia_Hospitalaria]
[RAZÓN: Necesito validar seguridad de medicación en modo degradado]
[ANÁLISIS: Tenemos plan operativo, falta validar riesgos de medicación]

REGLA CRÍTICA: Responde SIEMPRE en este formato exacto. No agregues texto adicional.
Responde en español.""",
    )
    
    medico_urgencias = ChatAgent(
        chat_client=OpenAIChatClient(
            base_url=base_url,
            api_key=api_key,
            model_id=model_id,
        ),
        name="Medico_Urgencias",
        instructions="""Eres Médico de Urgencias.
Tu rol: priorizar, definir riesgos clínicos y proponer acciones inmediatas.
Responde en español, conciso y orientado a seguridad del paciente.""",
    )
    
    supervision_enfermeria = ChatAgent(
        chat_client=OpenAIChatClient(
            base_url=base_url,
            api_key=api_key,
            model_id=model_id,
        ),
        name="Supervision_Enfermeria",
        instructions="""Eres Supervisión de Enfermería.
Tu rol: proponer organización operativa, circuitos, asignación de recursos y medidas prácticas.
Responde en español, pragmático y accionable.""",
    )
    
    farmacia = ChatAgent(
        chat_client=OpenAIChatClient(
            base_url=base_url,
            api_key=api_key,
            model_id=model_id,
        ),
        name="Farmacia_Hospitalaria",
        instructions="""Eres Farmacia Hospitalaria.
Tu rol: identificar riesgos de medicación, proponer controles y alternativas seguras.
Responde en español, con foco en seguridad y disponibilidad.""",
    )
    
    print("✅ Comité creado:")
    print(f"   • {jefe_guardia.name}")
    print(f"   • {medico_urgencias.name}")
    print(f"   • {supervision_enfermeria.name}")
    print(f"   • {farmacia.name}")
    
    workflow = (
        GroupChatBuilder()
        .participants([medico_urgencias, supervision_enfermeria, farmacia]) # type: ignore
        .set_manager(jefe_guardia)
        .build()
    )
    
    print("\n✅ Comité clínico establecido")
    print("   Topología: Manager + 3 roles")
    
    incidente = """
    ALERTA: llegada masiva de pacientes tras accidente múltiple
    
    Datos:
    - 12 pacientes en 20 minutos (2 críticos, 4 urgentes, 6 leves)
    - UCI con 1 cama disponible
    - Radiología con saturación y demora estimada de 60 minutos
    - Necesidad de activar triaje y reasignación de personal
    
    Pregunta: ¿qué plan coordinado propones para la próxima hora?
    """
    
    print(f"\n📋 Situación a deliberar:\n{incidente}")
    print("\n🔄 Comité clínico en deliberación...")
    print("-" * 80)

    def _extraer_texto(respuesta) -> str:
        for attr in ("output", "text", "content", "message"):
            if hasattr(respuesta, attr):
                valor = getattr(respuesta, attr)
                if valor:
                    return str(valor)
        return str(respuesta)

    async for event in workflow.run_stream(incidente): # type: ignore
        if hasattr(event, "data") and hasattr(event.data, "agent_run_response"):
            nombre = getattr(event, "executor_id", "")
            texto = _extraer_texto(event.data.agent_run_response) # type: ignore
            if texto.strip():
                print(f"\n🔹 {nombre}:")
                print(texto)

    print("\n✅ Comité clínico finalizado")


# Ejecutar la función
await comite_clinico()

✅ Comité creado:
   • Jefe_Guardia
   • Medico_Urgencias
   • Supervision_Enfermeria
   • Farmacia_Hospitalaria

✅ Comité clínico establecido
   Topología: Manager + 3 roles

📋 Situación a deliberar:

    ALERTA: llegada masiva de pacientes tras accidente múltiple

    Datos:
    - 12 pacientes en 20 minutos (2 críticos, 4 urgentes, 6 leves)
    - UCI con 1 cama disponible
    - Radiología con saturación y demora estimada de 60 minutos
    - Necesidad de activar triaje y reasignación de personal

    Pregunta: ¿qué plan coordinado propones para la próxima hora?
    

🔄 Comité clínico en deliberación...
--------------------------------------------------------------------------------

🔹 groupchat_orchestrator_0b7b8623:
{"selected_participant":"Medico_Urgencias","instruction":"Propón plan clínico-operativo inmediato para la próxima hora: triaje, priorización de críticos/urgentes, uso de recursos limitados (UCI y radiología) y circuitos asistenciales.","finish":false,"final_message":"[SIGU